In [ ]:
import matplotlib_inline.config
import numpy as np
import tensorflow_datasets as tfds
import tensorflow as tf

import matplotlib.pyplot as plt

# Download the 'cats_vs_dogs' dataset

In [ ]:
try:
    setattr(tfds.image_classification.cats_vs_dogs, '_URL', "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip")
except AttributeError:
    try:
        setattr(tfds.image.cats_vs_dogs, '_URL', "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip")
    except AttributeError:
        print("Could not set the _URL attribute. Please ensure tensorflow_datasets is installed correctly and check the module path for cats_vs_dogs.")
        # As a last resort, you might need to find the cats_vs_dogs.py file in your
        # tensorflow_datasets installation and modify the _URL variable directly,
        # but this is highly discouraged as it can be overwritten by updates.

ds_train, ds_info = tfds.load(
    'cats_vs_dogs',
    split='train',
    with_info=True,
    as_supervised=True,
)

In [ ]:
ds_train, ds_info = tfds.load(
    'cats_vs_dogs',
    split='train',
    with_info=True,
    as_supervised=True,
)

In [ ]:
class_name = ['cat', 'dog']

In [ ]:
ds_train_ = ds_train.shuffle(buffer_size=10000, reshuffle_each_iteration=False)
num_train_examples = int(0.7 * len(ds_train_))
ds_train = ds_train_.take(num_train_examples)
ds_test = ds_train_.skip(num_train_examples)

In [ ]:
IMG_SIZE = 28
BATCH_SIZE = 64

In [ ]:
def preprocess_image(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.GAUSSIAN)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    image = tf.reshape(image, [-1])
    return image, tf.one_hot(label, depth=2)

In [ ]:
ds_train = ds_train.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
ds_train = ds_train.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_test = ds_test.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(ds_test.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image[0].numpy().reshape((28, 28, 3)))
    plt.title(class_name[np.argmax(label[0])])
    plt.axis("off")

In [ ]:
# TODO change code to the newest Tensorflow API
# TODO 1: change import
from keras.models import Sequential
from keras.layers import Dense, Flatten


model = Sequential()
# TODO 2: add Input layer
# TODO 3: remove `input_shape` param
model.add(Dense(units=64, activation='relu', input_shape=(2352,)))
model.add(Dense(units=64, activation='relu'))
model.add(Dense(units=64, activation='relu'))
model.add(Dense(units=2, activation="softmax"))

model.compile(loss='categorical_crossentropy',
              metrics=["accuracy"])
model.summary()

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=["accuracy"])

In [ ]:
history = model.fit(ds_train, validation_data=ds_test, epochs=60)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

epoch_array = range(1, len(history.history['loss']) + 1)
loss_array = history.history['loss']
acc_array = history.history['accuracy']

max_loss = max(loss_array)
loss_array = np.array(loss_array) / max_loss

plt.plot(epoch_array, loss_array, label="loss")
plt.plot(epoch_array, acc_array, label="acc")
plt.xlabel("epochs")
plt.title("total loss and accuracy")

plt.legend()
plt.show()

In [ ]:
epoch_array = range(1, len(history.history['loss']) + 1)
val_loss_array = history.history['val_loss']
loss_array = history.history['loss']

plt.plot(epoch_array, loss_array, label="loss")
plt.plot(epoch_array, val_loss_array, label="val loss")
plt.xlabel("epochs")
plt.title("total loss and accuracy")

plt.legend()
plt.show()

In [ ]:
epoch_array = range(1, len(history.history['loss']) + 1)
val_acc_array = history.history['val_accuracy']
acc_array = history.history['accuracy']

plt.plot(epoch_array, acc_array, label="acc")
plt.plot(epoch_array, val_acc_array, label="val acc")
plt.xlabel("epochs")
plt.title("total loss and accuracy")

plt.legend()
plt.show()

In [ ]:
results = model.evaluate(ds_test)
print(results)

In [ ]:
from matplotlib import rcParams
from matplotlib import pyplot as plt

rcParams["figure.figsize"] = [10, 10]
rcParams['xtick.labelbottom'] = False

In [ ]:
# finally visualize it
x_test = np.concatenate([x for x, y in ds_test], axis=0)
y_test = np.concatenate([y for x, y in ds_test], axis=0)

test_pred = model.predict(x_test)

for idx, elem in enumerate(ds_test.take(25)):
    pred_idx = np.argmax(test_pred[idx])
    true_idx = np.argmax(y_test[idx])
    plt.subplot(5, 5, idx + 1, title=(class_name[pred_idx] + "(" + class_name[true_idx] + ")"))
    plt.imshow(elem[0][0].numpy().reshape((28, 28, 3)))

In [ ]:
from matplotlib import pyplot

test_pred = model.predict(x_test)
fig = pyplot.figure(figsize=(20, 8))

for idx, elem in enumerate(ds_test.take(32)):
    ax = fig.add_subplot(4, 8, idx + 1, xticks=[], yticks=[])
    ax.imshow(elem[0][0].numpy().reshape((28, 28, 3)))
    pred_idx = np.argmax(test_pred[idx])
    true_idx = np.argmax(y_test[idx])
    ax.set_title("{} ({})".format(class_name[pred_idx], class_name[true_idx]),
                 color=("green" if pred_idx == true_idx else "red"))
pyplot.show()